In [1]:
%pip install pandas numpy openpyxl scipy matplotlib

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np

print("Starting finalized data preparation process...")

# --------------------------------------------------------
# 1. Setup and Static Data Filtering
# --------------------------------------------------------
# Load the static dataset to identify the initial investment universe
static_df = pd.read_excel('Data_2026/Static_2025.xlsx')
static_df['ISIN'] = static_df['ISIN'].astype(str).str.strip()

# Isolate firms belonging to the Emerging Markets (EM) region
em_firms = static_df[static_df['Region'] == 'EM'].copy()
valid_em_isins = em_firms['ISIN'].unique().tolist()

def load_and_transpose(filepath, is_monthly=False):
    """
    Standardizes Datastream Excel exports into time-series DataFrames.
    Removes error flags ($$ER) and transposes firms to columns.
    """
    df = pd.read_excel(filepath)
    # Rule: Explicitly delete ISINs with Datastream errors ($$ER)
    df = df[~df['ISIN'].astype(str).str.contains('\$\$ER', na=False)]
    df['ISIN'] = df['ISIN'].astype(str).str.strip()
    df.set_index('ISIN', inplace=True)
    
    if 'NAME' in df.columns:
        df.drop(columns=['NAME'], inplace=True)
    
    df_t = df.T
    df_t = df_t.apply(pd.to_numeric, errors='coerce')
    
    # Handle indexing based on data frequency (Monthly for RI, Yearly for Carbon)
    if is_monthly:
        df_t.index = pd.to_datetime(df_t.index)
    else:
        df_t.index = df_t.index.astype(int)
    return df_t

# --------------------------------------------------------
# 2. Return Index (RI) Processing with Delisting Logic
# --------------------------------------------------------
ri_monthly = load_and_transpose('Data_2026/DS_RI_T_USD_M_2025.xlsx', is_monthly=True)
ri_em = ri_monthly[[isin for isin in valid_em_isins if isin in ri_monthly.columns]].copy()

# Rule: Treat prices < 0.5 as missing values to avoid extreme returns from penny stocks
ri_em[ri_em < 0.5] = np.nan

# CRITICAL: Identify the real last valid date BEFORE filling gaps.
# This ensures that delisting (final disappearance) is not masked by forward-filling.
real_last_valid_dates = ri_em.apply(lambda col: col.last_valid_index())

# Rule: Fill gaps between available values (Forward Fill) to bridge misreporting
ri_em_filled = ri_em.ffill()
returns_em = ri_em_filled.pct_change()

# Applying Precise Delisting Logic: -100% loss followed by NaNs
for isin in returns_em.columns:
    last_date = real_last_valid_dates[isin]
    if pd.notna(last_date):
        last_pos = returns_em.index.get_loc(last_date)
        
        # If the firm disappears before the end of the sample, acknowledge the loss
        if last_pos < len(returns_em) - 1:
            # Force -100% return in the month following the last valid price
            returns_em.iloc[last_pos + 1, returns_em.columns.get_loc(isin)] = -1.0
            
            # Ensure post-delisting periods are NaN (not zero) to avoid stale price/low volatility bias
            if last_pos + 1 < len(returns_em) - 1:
                returns_em.iloc[last_pos + 2:, returns_em.columns.get_loc(isin)] = np.nan
    else:
        # Mark firms with no valid data for complete removal
        returns_em[isin] = np.nan

# --------------------------------------------------------
# 3. Carbon Data Processing
# --------------------------------------------------------
scope1_em = load_and_transpose('Data_2026/DS_CO2_SCOPE_1_Y_2025.xlsx', is_monthly=False)
scope2_em = load_and_transpose('Data_2026/DS_CO2_SCOPE_2_Y_2025.xlsx', is_monthly=False)

# Filter for EM and apply Forward Fill for reporting gaps between years
scope1_em = scope1_em[[isin for isin in valid_em_isins if isin in scope1_em.columns]].copy().ffill()
scope2_em = scope2_em[[isin for isin in valid_em_isins if isin in scope2_em.columns]].copy().ffill()

# Rule: Physically delete firms with no associated data from all tables
returns_em.dropna(how='all', axis=1, inplace=True)
scope1_em.dropna(how='all', axis=1, inplace=True)
scope2_em.dropna(how='all', axis=1, inplace=True)

print("Data Validation Overview:")
print(f"- Returns Matrix: {returns_em.shape[0]} months, {returns_em.shape[1]} firms")
print(f"- Scope 1/2 Matrices: {scope1_em.shape[0]} years")
display(returns_em.head())

<>:24: SyntaxWarning: invalid escape sequence '\$'
<>:24: SyntaxWarning: invalid escape sequence '\$'
C:\Users\matte\AppData\Local\Temp\ipykernel_24160\3211466884.py:24: SyntaxWarning: invalid escape sequence '\$'
  df = df[~df['ISIN'].astype(str).str.contains('\$\$ER', na=False)]


Starting finalized data preparation process...
Data Validation Overview:
- Returns Matrix: 314 months, 668 firms
- Scope 1/2 Matrices: 27 years


ISIN,ARALUA010258,ARP125991090,ARSIDE010029,BMG211591018,BRABEVACNOR1,BRBBASACNOR3,BRBBDCACNPR8,BRBRFSACNOR8,BRBRKMACNPA4,BRCESPACNPB4,...,ZAE000117321,ZAE000127148,ZAE000134961,ZAE000170049,ZAE000179420,ZAE000191342,ZAE000255915,ZAE000298253,ZAE000302618,ZAE000322095
1999-12-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-31,0.176511,-0.051861,-0.004877,NaN,-0.044175,-0.067245,0.023986,-0.249634,0.011906,NaN,...,-0.115161,-0.049527,0.096965,0.067794,0.205596,0.099568,-0.080722,0.125341,0.021174,-0.114179
2000-02-29,0.050002,0.178923,0.039543,NaN,-0.045539,-0.036328,-0.107794,0.005452,0.043492,NaN,...,-0.037800,-0.100608,-0.326248,0.028928,-0.053481,-0.084315,-0.062844,-0.181598,0.018379,-0.023150
2000-03-31,-0.023552,-0.105072,-0.002113,NaN,0.313941,-0.130625,0.152061,0.014002,0.002361,NaN,...,-0.067887,-0.034040,-0.044066,-0.176088,-0.126866,0.073915,-0.054382,-0.038462,-0.142989,-0.104274
2000-04-28,-0.016173,-0.161842,-0.071188,NaN,-0.001350,-0.077311,-0.108617,-0.032966,-0.020143,NaN,...,-0.034586,-0.042240,-0.172342,-0.129131,-0.035409,-0.208833,-0.030503,-0.224615,-0.095572,0.003496


In [3]:
print("Initiating dynamic filtering for 10-year rolling windows...")

# Step 5: Dynamic Universe Construction
# We identify eligible firms for each year based on historical liquidity and ESG disclosure.
valid_investment_sets = {}
estimation_years = range(2009, 2025)

for Y in estimation_years:
    # Define the 10-year rolling estimation window (Dec Y-9 to Dec Y)
    start_date, end_date = f"{Y-9}-01-01", f"{Y}-12-31"
    window_returns = returns_em.loc[start_date:end_date]
    window_prices = ri_em.loc[start_date:end_date]
    
    valid_isins_for_year = []
    
    for isin in returns_em.columns:
        # Rule: Price Availability at end of year Y
        # Exclude firms missing a valid price (>0.5 USD) at the portfolio formation date.
        if pd.isna(window_prices[isin].iloc[-1]):
            continue
            
        # Rule: Stale Price / Liquidity Filter
        # Calculate the ratio of months with zero returns to avoid illiquid assets.
        total_valid_months = window_returns[isin].notna().sum()
        if total_valid_months == 0: continue
        
        # Exclude if the firm has no price movement for > 50% of the window.
        if (window_returns[isin] == 0.0).sum() / total_valid_months > 0.5:
            continue
            
        # Rule: Carbon Data Availability for Year Y
        # Both Scope 1 and Scope 2 data must be present for the firm to be eligible.
        if Y in scope1_em.index and Y in scope2_em.index:
            if isin in scope1_em.columns and isin in scope2_em.columns:
                if not pd.isna(scope1_em.loc[Y, isin]) and not pd.isna(scope2_em.loc[Y, isin]):
                    valid_isins_for_year.append(isin)
    
    # Store the investment set for the subsequent holding period (Y + 1)
    valid_investment_sets[Y + 1] = valid_isins_for_year
    print(f"Holding Year {Y+1}: {len(valid_isins_for_year)} firms identified.")

print("\nInvestment universes constructed successfully.")

Initiating dynamic filtering for 10-year rolling windows...
Holding Year 2010: 67 firms identified.
Holding Year 2011: 160 firms identified.
Holding Year 2012: 199 firms identified.
Holding Year 2013: 225 firms identified.
Holding Year 2014: 246 firms identified.
Holding Year 2015: 268 firms identified.
Holding Year 2016: 298 firms identified.
Holding Year 2017: 335 firms identified.
Holding Year 2018: 382 firms identified.
Holding Year 2019: 418 firms identified.
Holding Year 2020: 465 firms identified.
Holding Year 2021: 505 firms identified.
Holding Year 2022: 542 firms identified.
Holding Year 2023: 572 firms identified.
Holding Year 2024: 580 firms identified.
Holding Year 2025: 567 firms identified.

Investment universes constructed successfully.


In [4]:
# solo per noi per controllare:
# 1. vedere se in returns prima degli 0 finali ci sia un -1
# 2. vedere se in prices non ci siano valori < 0.5, io ho controllato facendo conditional formatting con i 10 piu
# bassi e sono tutti 0.5 quindi sembra ok
# non so se ci sta altro da controllare

with pd.ExcelWriter('Data_2026/FINAL_CHECK_CLEANING.xlsx') as writer:
    returns_em.iloc[:, :100].to_excel(writer, sheet_name='Returns_Check')
    ri_em.iloc[:, :100].to_excel(writer, sheet_name='Prices_Check')
    scope1_em.iloc[:, :100].to_excel(writer, sheet_name='Carbon_Check')

print("'FINAL_CHECK_CLEANING.xlsx' generated for verification.")

'FINAL_CHECK_CLEANING.xlsx' generated for verification.
